# OULAD Models Report

## Purpose

This notebook runs the scripts of data processing, documents the modeling for both baseline and advanced pipelines and performs the evaluation part of the OULAD project. The goal is to compare several pipelines for early student-risk prediction using week-based cutoffs from the engineered feature tables.

## What This Notebook Shows
- end-to-end pipeline execution To ensure full reproducibility without manual merging conflicts
- raw data extraction and week-based cutoffs and availability checks for the raw OULAD files
- leakage-safe feature engineering
- baseline model generation
- loading of previously saved reference pipelines
- tuning and benchmarking of advanced models across `week2`, `week4`, `week6`, and `week8`
- comparison using Accuracy, Precision, Recall, F1, AUC, and cross-validation statistics
- CV and ablation analysis to separate the effect of demographics and engagement features
- evaluation & KPIs as well as threshold & insights analysis
- optional saving of tuned artifacts for reproducibility
- error analysis


The notebook is written in report style so that the outputs can be discussed directly in a presentation or technical handoff.

In [1]:
from pathlib import Path
import zipfile

def find_project_root(required_paths):
    search_roots = [Path.cwd(), *Path.cwd().parents]
    candidates = search_roots + [root / 'ELEN4025_Group_Assignment' for root in search_roots]

    for candidate in candidates:
        if all((candidate / relative_path).exists() for relative_path in required_paths):
            return candidate

    missing = ', '.join(required_paths)
    raise FileNotFoundError(f"Could not find a project folder containing: {missing}")

project_dir = find_project_root(['anonymisedData.zip'])
zip_file_path = project_dir / 'anonymisedData.zip'
required_files = ['studentInfo.csv', 'studentVle.csv', 'vle.csv']

print(f"Using project directory: {project_dir}")

if all((project_dir / file_name).exists() for file_name in required_files):
    print("Datasets already extracted. Skipping extraction step.")
else:
    print("Extracting datasets ...")
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(project_dir)
    print("Extraction complete.")

Using project directory: c:\Users\User1\projects\ELEN4025-GROUP\ELEN4025_Group_Assignment
Datasets already extracted. Skipping extraction step.


In [ ]:
# Data Processing
print("Data Cutoffs...")
!python scripts/build_task2_cutoffs.py

print("Feature Engineering...")
!python scripts/build_task3_features.py

# Model Training
print("Baseline Models...")
!python scripts/run_task4_baseline_models.py

print("Select Advanced Models...")
!python scripts/train_pipeline5_catboost.py
!python scripts/train_pipeline_3_logreg_weighted.py

Data Cutoffs...


## Data Preparation Context

The project uses `anonymisedData.zip` as the raw source bundle. Earlier project stages produced the cutoff-specific files in `data/interim/` and the engineered feature tables in `data/processed/`.

For the advanced-model experiments, the key inputs are the processed tables `week2_features.csv`, `week4_features.csv`, `week6_features.csv`, and `week8_features.csv`. Each table contains one row per student-module-presentation record, demographic features, cumulative engagement features up to the selected cutoff, and the binary label used for prediction.

## Previously Saved Reference Pipelines

Before fitting tuned models, the notebook loads the supplied saved pipelines. These serve as reference artifacts from earlier experimentation and make it possible to compare the final tuned workflow against the versions that already exist in the repository.

In [ ]:
# Inspect saved models for any load errors
print("Inspecting saved models...")
!python scripts/inspect_saved_models.py

Inspecting saved models...

INSPECTING SAVED MODEL FILES

File: pipeline_1_rf_calibrated.pkl
Path: models\pipeline_1_rf_calibrated.pkl
Loaded successfully.
Object type: <class 'sklearn.pipeline.Pipeline'>
Has predict: True
Has predict_proba: True
Has decision_function: False
Pipeline steps:
  - preprocessor: ColumnTransformer
  - classifier: CalibratedClassifierCV
classes_: [0 1]
n_features_in_: 8
feature_names_in_ first 20: ['total_clicks_w4', 'active_days_w4', 'forum_clicks_w4', 'resource_clicks_w4', 'num_of_prev_attempts', 'gender', 'region', 'highest_education']
Key params: {'classifier__estimator__bootstrap': True, 'classifier__estimator__ccp_alpha': 0.0, 'classifier__estimator__class_weight': 'balanced', 'classifier__estimator__criterion': 'gini', 'classifier__estimator__max_depth': 10, 'classifier__estimator__max_features': 'sqrt', 'classifier__estimator__max_leaf_nodes': None, 'classifier__estimator__max_samples': None, 'classifier__estimator__min_impurity_decrease': 0.0, 'clas

c:\Users\User1\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\User1\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\User1\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.8.0 when using version 1.7.2. This might lead to breaki

In [ ]:
from pathlib import Path
import joblib
import warnings

def find_project_root(required_paths):
    search_roots = [Path.cwd(), *Path.cwd().parents]
    candidates = search_roots + [root / 'ELEN4025_Group_Assignment' for root in search_roots]

    for candidate in candidates:
        if all((candidate / relative_path).exists() for relative_path in required_paths):
            return candidate

    missing = ', '.join(required_paths)
    raise FileNotFoundError(f"Could not find a project folder containing: {missing}")

try:
    from sklearn.exceptions import InconsistentVersionWarning
except Exception:
    InconsistentVersionWarning = None

if InconsistentVersionWarning is not None:
    warnings.filterwarnings('ignore', category=InconsistentVersionWarning)

project_dir = find_project_root(['models'])
model_dir = project_dir / 'models'
pipelines = {}
load_errors = {}
missing_packages = set()

for model_file in sorted(model_dir.glob('*.pkl')):
    model_name = model_file.stem
    try:
        pipelines[model_name] = joblib.load(model_file)
    except ModuleNotFoundError as exc:
        missing_packages.add(exc.name)
        load_errors[model_name] = f"Missing package: {exc.name}"
    except Exception as exc:
        load_errors[model_name] = f"{type(exc).__name__}: {exc}"

print(f"Loaded {len(pipelines)} pipelines: {list(pipelines)}")

if load_errors:
    print("Skipped pipelines:")
    for model_name, error_message in load_errors.items():
        print(f" - {model_name}: {error_message}")

if missing_packages:
    package_list = ' '.join(sorted(missing_packages))
    print(f"Install missing libraries to load every saved model: pip install {package_list}")

Loaded 7 pipelines: ['pipeline_1_rf_calibrated', 'pipeline_2_gbm', 'pipeline_3_logreg_weighted_balanced', 'pipeline_4_rf_model-weighted_balance', 'pipeline_5_catboost', 'pipeline_6_knn', 'pipeline_7_xgboost']


## Advanced Model Benchmark

### Experimental Objective

The purpose of this section is to evaluate whether more advanced or self-learned models improve early risk prediction as more weeks of student activity become available.

### Models Included

- training the advanced pipelines across `week2`, `week4`, `week6`, and `week8`
- benchmarking Random Forest, Gradient Boosting, weighted Random Forest, CatBoost, KNN, and XGBoost
- tuning model hyperparameters with `RandomizedSearchCV`
- reporting Accuracy, Precision, Recall, F1, AUC, validation threshold, and cross-validation summaries
- identifying the best advanced model per cutoff
- saving reproducible comparison tables to `data/processed/`

### Evaluation Design

Each cutoff is evaluated separately. The workflow uses train, validation, and test splits, selects a classification threshold from the validation set, and then reports final holdout performance on the untouched test set. Cross-validation is also included to show whether performance is stable across folds rather than dependent on one random split.

The full benchmark is compute-intensive. You can reduce `SEARCH_ITERATIONS`, `CV_SPLITS`, `BENCHMARK_WEEKS`, or `BENCHMARK_MODELS` in the config cell for quicker experiments.

In [ ]:
if 'find_project_root' not in globals():
    from pathlib import Path

    def find_project_root(required_paths):
        search_roots = [Path.cwd(), *Path.cwd().parents]
        candidates = search_roots + [root / 'ELEN4025_Group_Assignment' for root in search_roots]

        for candidate in candidates:
            if all((candidate / relative_path).exists() for relative_path in required_paths):
                return candidate

        missing = ', '.join(required_paths)
        raise FileNotFoundError(f"Could not find a project folder containing: {missing}")

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from IPython.display import display
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, make_scorer, precision_recall_curve, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

BENCHMARK_WEEKS = ['week2', 'week4', 'week6', 'week8']
BENCHMARK_MODELS = [
    'pipeline_1_rf_calibrated.pkl',
    'pipeline_2_gbm.pkl',
    'pipeline_3_logreg_weighted_balanced.pkl',
    'pipeline_4_rf_model-weighted_balance.pkl',
    'pipeline_5_catboost.pkl',
    'pipeline_6_knn.pkl',
    'pipeline_7_xgboost.pkl',
]

SEARCH_ITERATIONS = 4
CV_SPLITS = 3
TEST_SIZE = 0.2
VALIDATION_SIZE = 0.25
RANDOM_STATE = 42

SAVE_BENCHMARK_TABLES = True
SAVE_TRAINED_MODELS = False

project_dir = find_project_root(['data/processed', 'models'])
processed_dir = project_dir / 'data' / 'processed'
model_output_root = project_dir / 'models'

benchmark_output_path = processed_dir / 'advanced_model_benchmark.csv'
best_model_output_path = processed_dir / 'advanced_model_best_per_week.csv'
ablation_output_path = processed_dir / 'advanced_model_ablation.csv'

print(f'Project directory: {project_dir}')
print(f'Benchmark weeks: {BENCHMARK_WEEKS}')
print(f'Benchmark models: {BENCHMARK_MODELS}')
print(f'Search iterations per model: {SEARCH_ITERATIONS}')
print(f'Cross-validation folds: {CV_SPLITS}')

Project directory: c:\Users\User1\projects\ELEN4025-GROUP\ELEN4025_Group_Assignment
Benchmark weeks: ['week2', 'week4', 'week6', 'week8']
Benchmark models: ['pipeline_1_rf_calibrated.pkl', 'pipeline_2_gbm.pkl', 'pipeline_3_logreg_weighted_balanced.pkl', 'pipeline_4_rf_model-weighted_balance.pkl', 'pipeline_5_catboost.pkl', 'pipeline_6_knn.pkl', 'pipeline_7_xgboost.pkl']
Search iterations per model: 4
Cross-validation folds: 3


In [ ]:
ID_COLUMNS = ['id_student', 'code_module', 'code_presentation']
TARGET_COLUMN = 'label'

KNOWN_CATEGORICALS = [
    'gender',
    'region',
    'highest_education',
    'imd_band',
    'age_band',
    'disability',
    'code_module',
    'code_presentation',
]

SCORING = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, zero_division=0),
    'recall': make_scorer(recall_score, zero_division=0),
    'f1': make_scorer(f1_score, zero_division=0),
    'roc_auc': 'roc_auc',
}


def default_catboost_params():
    return {
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'auto_class_weights': 'Balanced',
        'random_state': RANDOM_STATE,
        'verbose': False,
    }


def json_safe(value):
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, dict):
        return {key: json_safe(item) for key, item in value.items()}
    return value


def serialize_params(params):
    return json.dumps(json_safe(params), sort_keys=True)


def get_search_iterations(search_space):
    total = 1
    for values in search_space.values():
        total *= len(values)
    return max(1, min(SEARCH_ITERATIONS, total))


def load_week_dataset(week):
    dataset_path = processed_dir / f'{week}_features.csv'
    dataset = pd.read_csv(dataset_path)
    return dataset


def make_preprocessor(numeric_columns, categorical_columns):
    transformers = []

    if numeric_columns:
        transformers.append(
            (
                'num',
                Pipeline(
                    steps=[
                        ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
                        ('scaler', StandardScaler()),
                    ]
                ),
                numeric_columns,
            )
        )

    if categorical_columns:
        transformers.append(
            (
                'cat',
                Pipeline(
                    steps=[
                        ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
                        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
                    ]
                ),
                categorical_columns,
            )
        )

    return ColumnTransformer(transformers=transformers, remainder='drop')


def prepare_catboost_frame(frame, categorical_columns):
    prepared = frame.copy()
    for column in categorical_columns:
        if isinstance(column, int):
            column = prepared.columns[column]
        if column not in prepared.columns:
            continue
        prepared[column] = prepared[column].fillna('Missing').astype(str)
    return prepared

def get_catboost_feature_indices(feature_columns, categorical_columns):
    feature_positions = {column: index for index, column in enumerate(feature_columns)}
    return [
        feature_positions[column]
        for column in categorical_columns
        if column in feature_positions
    ]

def is_categorical_feature(column, series):
    return column in KNOWN_CATEGORICALS or not pd.api.types.is_numeric_dtype(series)

def get_feature_plan(week, feature_columns, X_frame):
    full_categorical_columns = [column for column in feature_columns if is_categorical_feature(column, X_frame[column])]
    full_numeric_columns = [column for column in feature_columns if column not in full_categorical_columns]

    base_numeric_columns = [
        column
        for column in [
            f'total_clicks_{week}',
            f'active_days_{week}',
            f'forumng_clicks_{week}',
            f'resource_clicks_{week}',
            'num_of_prev_attempts',
        ]
        if column in feature_columns
    ]
    base_categorical_columns = [
        column for column in ['gender', 'region', 'highest_education'] if column in feature_columns
    ]

    alt_numeric_columns = [
        column
        for column in [
            f'total_clicks_{week}',
            f'active_days_{week}',
            f'forumng_clicks_{week}',
            'studied_credits',
            'num_of_prev_attempts',
        ]
        if column in feature_columns
    ]
    alt_categorical_columns = [column for column in ['imd_band'] if column in feature_columns]

    engagement_only = [column for column in feature_columns if column.endswith(f'_{week}')]
    demographics_only = [column for column in feature_columns if column not in engagement_only]

    return {
        'feature_columns': feature_columns,
        'full_categorical_columns': full_categorical_columns,
        'cat_feature_indices': full_categorical_columns,  # Using the same list for CatBoost categorical features
        'full_numeric_columns': full_numeric_columns,
        'base_numeric_columns': base_numeric_columns,
        'base_categorical_columns': base_categorical_columns,
        'alt_numeric_columns': alt_numeric_columns,
        'alt_categorical_columns': alt_categorical_columns,
        'engagement_only': engagement_only,
        'demographics_only': demographics_only,
    }


def make_week_split(dataset, week):
    feature_columns = [column for column in dataset.columns if column not in ID_COLUMNS + [TARGET_COLUMN]]
    X = dataset[feature_columns].copy()
    y = dataset[TARGET_COLUMN].astype(int)

    for col in KNOWN_CATEGORICALS:
        if col in X.columns:
            X[col] = X[col].astype(str)


    feature_plan = get_feature_plan(week, feature_columns, X)

    X_dev, X_test, y_dev, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_dev,
        y_dev,
        test_size=VALIDATION_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_dev,
    )

    return {
        'X': X,
        'y': y,
        'feature_columns': feature_columns,
        'feature_plan': feature_plan,
        'X_train': X_train,
        'X_val': X_val,
        'X_dev': X_dev,
        'X_test': X_test,
        'y_train': y_train,
        'y_val': y_val,
        'y_dev': y_dev,
        'y_test': y_test,
    }


def build_model_specs(week, split_data):
    plan = split_data['feature_plan']

    return {
        'pipeline_1_rf_calibrated.pkl': {
            'kind': 'sklearn',
            'estimator': Pipeline(
                steps=[
                    ('preprocessor', make_preprocessor(plan['base_numeric_columns'], plan['base_categorical_columns'])),
                    (
                        'classifier',
                        CalibratedClassifierCV(
                            estimator=RandomForestClassifier(
                                class_weight='balanced',
                                random_state=RANDOM_STATE,
                                n_jobs=1,
                            ),
                            method='sigmoid',
                            cv=3,
                        ),
                    ),
                ]
            ),
            'search_space': {
                'classifier__estimator__n_estimators': [200, 400],
                'classifier__estimator__max_depth': [8, 12, None],
                'classifier__estimator__min_samples_leaf': [1, 2, 4],
                'classifier__estimator__min_samples_split': [2, 5, 10],
            },
        },
        'pipeline_2_gbm.pkl': {
            'kind': 'sklearn',
            'estimator': Pipeline(
                steps=[
                    ('preprocessor', make_preprocessor(plan['base_numeric_columns'], plan['base_categorical_columns'])),
                    ('classifier', GradientBoostingClassifier(random_state=RANDOM_STATE)),
                ]
            ),
            'search_space': {
                'classifier__n_estimators': [150, 250, 350],
                'classifier__learning_rate': [0.03, 0.05, 0.1],
                'classifier__max_depth': [2, 3, 4],
                'classifier__subsample': [0.8, 1.0],
            },
        },
        'pipeline_3_logreg_weighted_balanced.pkl': {
            'kind': 'sklearn',
            'estimator': Pipeline(
                steps=[
                    ('preprocessor', make_preprocessor(plan['base_numeric_columns'], plan['base_categorical_columns'])),
                    (
                        'classifier',
                        LogisticRegression(
                            class_weight='balanced',
                            max_iter=1000,
                            random_state=RANDOM_STATE
                        )
                    )
                ]
            ),
            'search_space': {
                'classifier__C': [0.01, 0.1, 1.0, 10.0],
                'classifier__solver': ['lbfgs', 'liblinear']
            },
        },
        'pipeline_4_rf_model-weighted_balance.pkl': {
            'kind': 'sklearn',
            'estimator': Pipeline(
                steps=[
                    ('preprocessor', make_preprocessor(plan['base_numeric_columns'], plan['base_categorical_columns'])),
                    (
                        'classifier',
                        RandomForestClassifier(
                            class_weight='balanced',
                            random_state=RANDOM_STATE,
                            n_jobs=1,
                        ),
                    ),
                ]
            ),
            'search_space': {
                'classifier__n_estimators': [250, 500],
                'classifier__max_depth': [10, 16, None],
                'classifier__min_samples_leaf': [1, 2, 4],
                'classifier__min_samples_split': [2, 5, 10],
            },
        },
        'pipeline_5_catboost.pkl': {
            'kind': 'catboost',
            'estimator': CatBoostClassifier(**default_catboost_params()),
            'feature_columns': plan['feature_columns'],
            'cat_features': plan['cat_feature_indices'],  # Using indices for CatBoost categorical features
            'cat_feature_indices': get_catboost_feature_indices(
                plan['feature_columns'],
                plan['cat_feature_indices'],
            ),
            'search_space': {
                'depth': [4, 6, 8],
                'learning_rate': [0.03, 0.05, 0.1],
                'iterations': [300, 500, 700],
                'l2_leaf_reg': [1, 3, 5, 7],
            },
        },
        'pipeline_6_knn.pkl': {
            'kind': 'sklearn',
            'estimator': Pipeline(
                steps=[
                    ('preprocessor', make_preprocessor(plan['alt_numeric_columns'], plan['alt_categorical_columns'])),
                    ('classifier', KNeighborsClassifier()),
                ]
            ),
            'search_space': {
                'classifier__n_neighbors': [11, 21, 31],
                'classifier__weights': ['uniform', 'distance'],
                'classifier__p': [1, 2],
            },
        },
        'pipeline_7_xgboost.pkl': {
            'kind': 'sklearn',
            'estimator': Pipeline(
                steps=[
                    ('preprocessor', make_preprocessor(plan['alt_numeric_columns'], plan['alt_categorical_columns'])),
                    (
                        'classifier',
                        XGBClassifier(
                            eval_metric='logloss',
                            random_state=RANDOM_STATE,
                            n_jobs=1,
                        ),
                    ),
                ]
            ),
            'search_space': {
                'classifier__n_estimators': [150, 300, 500],
                'classifier__max_depth': [3, 4, 6],
                'classifier__learning_rate': [0.03, 0.05, 0.1],
                'classifier__subsample': [0.8, 0.9, 1.0],
                'classifier__colsample_bytree': [0.8, 0.9, 1.0],
            },
        },
    }


def select_best_threshold(y_true, probabilities):
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
    if thresholds.size == 0:
        threshold = 0.5
        f1_value = f1_score(y_true, (probabilities >= threshold).astype(int), zero_division=0)
        return float(threshold), float(f1_value)

    numerator = 2 * precision[:-1] * recall[:-1]
    denominator = np.clip(precision[:-1] + recall[:-1], 1e-12, None)
    f1_scores = numerator / denominator
    best_index = int(np.nanargmax(f1_scores))
    return float(thresholds[best_index]), float(f1_scores[best_index])


def evaluate_probabilities(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    return {
        'accuracy': float(accuracy_score(y_true, predictions)),
        'precision': float(precision_score(y_true, predictions, zero_division=0)),
        'recall': float(recall_score(y_true, predictions, zero_division=0)),
        'f1': float(f1_score(y_true, predictions, zero_division=0)),
        'auc': float(roc_auc_score(y_true, probabilities)),
    }


def summarize_cv_results(cv_results):
    summary = {}
    for metric_name in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
        values = cv_results[f'test_{metric_name}']
        summary[f'cv_{metric_name}_mean'] = float(np.mean(values))
        summary[f'cv_{metric_name}_std'] = float(np.std(values, ddof=0))
    return summary


def predict_probabilities(spec, estimator, X_frame):
    if spec['kind'] == 'catboost':
        categorical_features = spec.get('cat_features', spec.get('cat_feature_indices', []))
        prepared = prepare_catboost_frame(X_frame[spec['feature_columns']], categorical_features)
        probabilities = estimator.predict_proba(prepared)[:, 1]
    else:
        return estimator.predict_proba(X_frame)[:, 1]
        probabilities = estimator.predict_proba(X_frame)[:, 1]
    return probabilities


def tune_estimator(spec, X_train, y_train):
    cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    n_iter = get_search_iterations(spec['search_space'])

    if spec['kind'] == 'catboost':
        prepared = prepare_catboost_frame(X_train[spec['feature_columns']], spec['cat_feature_indices'])
        search = RandomizedSearchCV(
            estimator=spec['estimator'],
            param_distributions=spec['search_space'],
            n_iter=n_iter,
            scoring='roc_auc',
            cv=cv,
            n_jobs=1,
            random_state=RANDOM_STATE,
            refit=True,
            error_score='raise',
        )
        search.fit(prepared, y_train, cat_features=spec['cat_feature_indices'], verbose=False)
        return search.best_estimator_, search.best_params_, float(search.best_score_)

    search = RandomizedSearchCV(
        estimator=spec['estimator'],
        param_distributions=spec['search_space'],
        n_iter=n_iter,
        scoring='roc_auc',
        cv=cv,
        n_jobs=1,
        random_state=RANDOM_STATE,
        refit=True,
        error_score='raise',
    )
    search.fit(X_train, y_train)
    return search.best_estimator_, search.best_params_, float(search.best_score_)


def cross_validate_estimator(spec, estimator, X_dev, y_dev):
    cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    if spec['kind'] != 'catboost':
        cv_results = cross_validate(
            clone(estimator),
            X_dev,
            y_dev,
            cv=cv,
            scoring=SCORING,
            n_jobs=1,
            error_score='raise',
        )
        return summarize_cv_results(cv_results)

    fold_rows = []
    estimator_params = estimator.get_params()
    for train_index, valid_index in cv.split(X_dev, y_dev):
        fold_model = CatBoostClassifier(**estimator_params)
        X_fold_train = prepare_catboost_frame(X_dev.iloc[train_index][spec['feature_columns']], spec['cat_feature_indices'])
        X_fold_valid = prepare_catboost_frame(X_dev.iloc[valid_index][spec['feature_columns']], spec['cat_feature_indices'])
        y_fold_train = y_dev.iloc[train_index]
        y_fold_valid = y_dev.iloc[valid_index]

        fold_model.fit(X_fold_train, y_fold_train, cat_features=spec['cat_feature_indices'], verbose=False)
        fold_probabilities = fold_model.predict_proba(X_fold_valid)[:, 1]
        fold_rows.append(evaluate_probabilities(y_fold_valid, fold_probabilities, 0.5))

    cv_frame = pd.DataFrame(fold_rows)
    return {
        'cv_accuracy_mean': float(cv_frame['accuracy'].mean()),
        'cv_accuracy_std': float(cv_frame['accuracy'].std(ddof=0)),
        'cv_precision_mean': float(cv_frame['precision'].mean()),
        'cv_precision_std': float(cv_frame['precision'].std(ddof=0)),
        'cv_recall_mean': float(cv_frame['recall'].mean()),
        'cv_recall_std': float(cv_frame['recall'].std(ddof=0)),
        'cv_f1_mean': float(cv_frame['f1'].mean()),
        'cv_f1_std': float(cv_frame['f1'].std(ddof=0)),
        'cv_roc_auc_mean': float(cv_frame['auc'].mean()),
        'cv_roc_auc_std': float(cv_frame['auc'].std(ddof=0)),
    }


def refit_estimator(spec, best_params, X_dev, y_dev):
    if spec['kind'] == 'catboost':
        params = spec['estimator'].get_params()
        params.update(best_params)
        final_model = CatBoostClassifier(**params)
        prepared = prepare_catboost_frame(X_dev[spec['feature_columns']], spec['cat_feature_indices'])
        final_model.fit(prepared, y_dev, cat_features=spec['cat_feature_indices'], verbose=False)
        return final_model

    final_model = clone(spec['estimator']).set_params(**best_params)
    final_model.fit(X_dev, y_dev)
    return final_model

## Benchmark Execution And Reading Guide

The next cell executes the full benchmarking loop for the selected advanced models and week cutoffs.

### Main Deliverables Produced By The Cell

- `final_comparison_table`, which is the main report-style comparison table
- `best_models_display`, which identifies the strongest advanced model for each cutoff
- optional CSV outputs for direct inclusion in the final project submission

### The Results

The `Precision`, `Recall`, and `F1` describe the quality of at-risk predictions, `AUC` captures ranking strength across thresholds, and the `CV` columns show whether the model remains stable across multiple folds instead of performing well on only one split.

In [ ]:
benchmark_rows = []
trained_artifacts = {}
week_splits = {}

for week in BENCHMARK_WEEKS:
    print(f'\n=== Benchmarking {week} ===')
    dataset = load_week_dataset(week)
    split_data = make_week_split(dataset, week)
    week_splits[week] = split_data
    model_specs = build_model_specs(week, split_data)

    print(f"Rows: {len(dataset):,} | Features: {len(split_data['feature_columns'])}")

    for model_name in BENCHMARK_MODELS:
        print(f'  Tuning and evaluating {model_name} ...')
        spec = model_specs[model_name]

        best_estimator, best_params, tuning_cv_auc = tune_estimator(spec, split_data['X_train'], split_data['y_train'])
        validation_probabilities = predict_probabilities(spec, best_estimator, split_data['X_val'])
        selected_threshold, validation_f1 = select_best_threshold(split_data['y_val'], validation_probabilities)
        test_probabilities = predict_probabilities(spec, best_estimator, split_data['X_test'])
        holdout_metrics = evaluate_probabilities(split_data['y_test'], test_probabilities, selected_threshold)
        cv_summary = cross_validate_estimator(spec, best_estimator, split_data['X_dev'], split_data['y_dev'])
        final_estimator = refit_estimator(spec, best_params, split_data['X_dev'], split_data['y_dev'])

        benchmark_rows.append(
            {
                'week': week,
                'model': model_name,
                'best_params': serialize_params(best_params),
                'selected_threshold': float(selected_threshold),
                'validation_f1': float(validation_f1),
                'tuning_cv_auc': float(tuning_cv_auc),
                **holdout_metrics,
                **cv_summary,
            }
        )

        trained_artifacts[(week, model_name)] = {
            'kind': spec['kind'],
            'estimator': final_estimator,
            'threshold': float(selected_threshold),
            'best_params': json_safe(best_params),
            'feature_columns': split_data['feature_columns'],
            'cat_features': spec.get('cat_features', []),
        }

benchmark_results = pd.DataFrame(benchmark_rows).sort_values(['week', 'auc', 'f1'], ascending=[True, False, False]).reset_index(drop=True)

final_comparison_table = benchmark_results[
    [
        'week',
        'model',
        'best_params',
        'selected_threshold',
        'accuracy',
        'precision',
        'recall',
        'f1',
        'auc',
        'cv_accuracy_mean',
        'cv_precision_mean',
        'cv_recall_mean',
        'cv_f1_mean',
        'cv_roc_auc_mean',
    ]
].rename(
    columns={
        'week': 'Week',
        'model': 'Model',
        'best_params': 'Best params',
        'selected_threshold': 'Threshold',
        'accuracy': 'Accuracy',
        'precision': 'Precision',
        'recall': 'Recall',
        'f1': 'F1',
        'auc': 'AUC',
        'cv_accuracy_mean': 'CV Accuracy',
        'cv_precision_mean': 'CV Precision',
        'cv_recall_mean': 'CV Recall',
        'cv_f1_mean': 'CV F1',
        'cv_roc_auc_mean': 'CV AUC',
    }
)

best_models_per_week = benchmark_results.sort_values(['week', 'auc', 'f1'], ascending=[True, False, False]).groupby('week', as_index=False).first()
best_models_per_week['why_best'] = best_models_per_week.apply(
    lambda row: f"Highest holdout AUC ({row['auc']:.3f}) with F1 {row['f1']:.3f} at threshold {row['selected_threshold']:.2f}",
    axis=1,
)

best_models_display = best_models_per_week[
    ['week', 'model', 'best_params', 'accuracy', 'precision', 'recall', 'f1', 'auc', 'cv_roc_auc_mean', 'why_best']
].rename(
    columns={
        'week': 'Week',
        'model': 'Best model',
        'best_params': 'Best params',
        'accuracy': 'Accuracy',
        'precision': 'Precision',
        'recall': 'Recall',
        'f1': 'F1',
        'auc': 'AUC',
        'cv_roc_auc_mean': 'CV AUC',
        'why_best': 'Why this model won',
    }
)

display(final_comparison_table)
display(best_models_display)

if SAVE_BENCHMARK_TABLES:
    final_comparison_table.to_csv(benchmark_output_path, index=False)
    best_models_display.to_csv(best_model_output_path, index=False)
    print(f'Saved benchmark table to {benchmark_output_path}')
    print(f'Saved best-per-week table to {best_model_output_path}')


=== Benchmarking week2 ===
Rows: 32,593 | Features: 18
  Tuning and evaluating pipeline_1_rf_calibrated.pkl ...
  Tuning and evaluating pipeline_2_gbm.pkl ...
  Tuning and evaluating pipeline_3_logreg_weighted_balanced.pkl ...
  Tuning and evaluating pipeline_4_rf_model-weighted_balance.pkl ...
  Tuning and evaluating pipeline_5_catboost.pkl ...


## Threshold Sweep Across Models

The threshold sweep is presented for all benchmark models at `week8` in order to compare how the decision threshold affects classification behaviour when the largest amount of student engagement information is available. Since week 8 is the latest evaluated cutoff, it provides the most complete view of student activity and is therefore the most suitable point for comparing threshold behaviour across models.

For each model, the first graph shows how precision, recall, and F1 vary across candidate thresholds on the validation set, while the second graph shows the corresponding precision-recall curve with the selected threshold highlighted. Together, these plots illustrate how different models trade off between correctly identifying at-risk students and limiting false alarms.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import precision_recall_curve

WEEK_FOR_SWEEP = 'week8'
MODELS_FOR_SWEEP = BENCHMARK_MODELS.copy()

dataset = load_week_dataset(WEEK_FOR_SWEEP)
split_data = make_week_split(dataset, WEEK_FOR_SWEEP)
model_specs = build_model_specs(WEEK_FOR_SWEEP, split_data)

for model_name in MODELS_FOR_SWEEP:
    spec = model_specs[model_name]

    best_estimator, best_params, tuning_cv_auc = tune_estimator(
        spec,
        split_data['X_train'],
        split_data['y_train'],
    )

    validation_probabilities = predict_probabilities(
        spec,
        best_estimator,
        split_data['X_val'],
    )

    selected_threshold, validation_f1 = select_best_threshold(
        split_data['y_val'],
        validation_probabilities,
    )

    precision, recall, thresholds = precision_recall_curve(
        split_data['y_val'],
        validation_probabilities,
    )

    if thresholds.size == 0:
        continue

    precision_t = precision[:-1]
    recall_t = recall[:-1]
    f1_t = (
        2 * precision_t * recall_t
        / np.clip(precision_t + recall_t, 1e-12, None)
    )

    selected_idx = int(np.argmin(np.abs(thresholds - selected_threshold)))
    plot_name = model_name.replace('.pkl', '')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(thresholds, precision_t, label='Precision')
    axes[0].plot(thresholds, recall_t, label='Recall')
    axes[0].plot(thresholds, f1_t, label='F1', linewidth=2)
    axes[0].axvline(
        selected_threshold,
        color='red',
        linestyle='--',
        label=f'Selected threshold = {selected_threshold:.3f}',
    )
    axes[0].scatter(
        thresholds[selected_idx],
        f1_t[selected_idx],
        color='red',
        zorder=3,
    )
    axes[0].set_xlabel('Threshold')
    axes[0].set_ylabel('Score')
    axes[0].set_title(f'{WEEK_FOR_SWEEP} - {plot_name}\nThreshold vs Precision / Recall / F1')
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(recall, precision, color='tab:blue')
    axes[1].scatter(
        recall_t[selected_idx],
        precision_t[selected_idx],
        color='red',
        label=f'Selected threshold = {selected_threshold:.3f}',
        zorder=3,
    )
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].set_title(f'{WEEK_FOR_SWEEP} - {plot_name}\nPrecision-Recall Curve')
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

## Ablation Study

### Motivation

Overall benchmark performance tells us which model works best, but it does not show which feature groups are responsible for the gain. The ablation study addresses this by holding the model family fixed and changing only the available inputs.

This ablation uses CatBoost across all cutoffs to compare three feature groups:

- `demographics_only`
- `engagement_only`
- `combined_all`

In presentation terms, this section answers a practical question: how much predictive value comes from weekly engagement behaviour beyond static background information?

In [ ]:
if 'benchmark_results' not in globals():
    raise RuntimeError('Run the advanced benchmark cell before the ablation cell.')

ablation_rows = []

for week in BENCHMARK_WEEKS:
    print(f'\n=== Ablation for {week} ===')
    split_data = week_splits.get(week)
    if split_data is None:
        split_data = make_week_split(load_week_dataset(week), week)

    catboost_row = benchmark_results[
        (benchmark_results['week'] == week) & (benchmark_results['model'] == 'pipeline_5_catboost.pkl')
    ].iloc[0]
    catboost_params = json.loads(catboost_row['best_params'])

    feature_sets = {
        'demographics_only': split_data['feature_plan']['demographics_only'],
        'engagement_only': split_data['feature_plan']['engagement_only'],
        'combined_all': split_data['feature_columns'],
    }

    for feature_set_name, selected_columns in feature_sets.items():
        print(f'  Training CatBoost ablation: {feature_set_name}')
        selected_columns = list(selected_columns)
        categorical_columns = [
           column
            for column in selected_columns
            if is_categorical_feature(column, split_data['X'][column])
            ]
        cat_feature_indices = get_catboost_feature_indices(selected_columns, categorical_columns)

        ablation_model = CatBoostClassifier(**{**default_catboost_params(), **catboost_params})
        X_train = prepare_catboost_frame(split_data['X_train'][selected_columns], categorical_columns)
        X_val = prepare_catboost_frame(split_data['X_val'][selected_columns], categorical_columns)
        X_test = prepare_catboost_frame(split_data['X_test'][selected_columns], categorical_columns)

        ablation_model.fit(X_train, split_data['y_train'], cat_features=cat_feature_indices, verbose=False)
        validation_probabilities = ablation_model.predict_proba(X_val)[:, 1]
        selected_threshold, validation_f1 = select_best_threshold(split_data['y_val'], validation_probabilities)
        test_probabilities = ablation_model.predict_proba(X_test)[:, 1]
        holdout_metrics = evaluate_probabilities(split_data['y_test'], test_probabilities, selected_threshold)

        ablation_rows.append(
            {
                'week': week,
                'feature_set': feature_set_name,
                'feature_count': len(selected_columns),
                'selected_threshold': float(selected_threshold),
                'validation_f1': float(validation_f1),
                **holdout_metrics,
            }
        )

ablation_results = pd.DataFrame(ablation_rows).sort_values(['week', 'auc', 'f1'], ascending=[True, False, False]).reset_index(drop=True)
ablation_display = ablation_results.rename(
    columns={
        'week': 'Week',
        'feature_set': 'Feature set',
        'feature_count': 'Feature count',
        'selected_threshold': 'Threshold',
        'validation_f1': 'Validation F1',
        'accuracy': 'Accuracy',
        'precision': 'Precision',
        'recall': 'Recall',
        'f1': 'F1',
        'auc': 'AUC',
    }
)

display(ablation_display)

if SAVE_BENCHMARK_TABLES:
    ablation_display.to_csv(ablation_output_path, index=False)
    print(f'Saved ablation table to {ablation_output_path}')

In [ ]:
# Execute CV/Ablation Notebook to generate graphs
#%run task6_cv_ablation.ipynb

# Task 6 Cross-Validation and Ablation Extension

The following cells are appended from `task6_cv_ablation_new(2).ipynb` as a separate Task 6 block. The original Team 4 notebook order is otherwise preserved.

## 1. Imports and global settings

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    BaggingClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    make_scorer,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

try:
    from sklearn.calibration import CalibratedClassifierCV
    HAS_CALIBRATION = True
except Exception:
    HAS_CALIBRATION = False

try:
    from sklearn.preprocessing import TargetEncoder
    HAS_TARGET_ENCODER = True
except Exception:
    HAS_TARGET_ENCODER = False

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except Exception:
    HAS_CATBOOST = False

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
CV_FOLDS = 5

WEEKS = ["week2", "week4", "week6", "week8"]
WEEK_NUM = {"week2": 2, "week4": 4, "week6": 6, "week8": 8}

TARGET = "label"
ID_COLS = ["id_student", "code_module", "code_presentation"]

DEMO_CAT_COLS = ["gender", "region", "highest_education", "imd_band", "age_band", "disability"]
DEMO_NUM_COLS = ["num_of_prev_attempts", "studied_credits"]

VLE_ACT_TYPES = [
    "oucontent", "quiz", "resource", "homepage",
    "subpage", "glossary", "oucollaborate", "forumng",
]

# This is False to align with the latest Task 4 script, which drops
# id_student, code_module, and code_presentation before modelling.

INCLUDE_MODULE_PRESENTATION_AS_FEATURES = False

Path("results").mkdir(exist_ok=True)

print("XGBoost available:", HAS_XGBOOST)
print("CatBoost available:", HAS_CATBOOST)
print("TargetEncoder available:", HAS_TARGET_ENCODER)


## 2. Locate project files

In [ ]:
def first_existing(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return None

def find_feature_file(week: str) -> Path:
    candidates = [
        Path("data") / "processed" / f"{week}_features.csv",
        Path(f"{week}_features.csv"),
        Path("processed") / f"{week}_features.csv",
    ]
    path = first_existing(candidates)
    if path is None:
        raise FileNotFoundError(f"Could not find {week}_features.csv")
    return path

task4_results_path = first_existing([
    "data/processed/task4_baseline_results.csv",
    "task4_baseline_results.csv",
])

model_inspection_path = first_existing([
    "data/processed/model_inspection_summary.csv",
    "model_inspection_summary.csv",
])

print("Task 4 results:", task4_results_path)
print("Model inspection summary:", model_inspection_path)

for week in WEEKS:
    print(week, "->", find_feature_file(week))


## 3. Load the week feature tables

In [ ]:
def load_week_df(week: str) -> pd.DataFrame:
    df = pd.read_csv(find_feature_file(week))
    if TARGET not in df.columns:
        raise ValueError(f"{week} file does not contain target column: {TARGET}")
    return df

def split_xy(df: pd.DataFrame):
    drop_cols = [TARGET, "id_student"]
    if not INCLUDE_MODULE_PRESENTATION_AS_FEATURES:
        drop_cols += ["code_module", "code_presentation"]
    drop_cols = [c for c in drop_cols if c in df.columns]

    X = df.drop(columns=drop_cols)
    y = df[TARGET].astype(int)
    return X, y

summary_rows = []
for week in WEEKS:
    df = load_week_df(week)
    X, y = split_xy(df)
    summary_rows.append({
        "week": week,
        "rows": len(df),
        "features_used": X.shape[1],
        "favourable_1": int((y == 1).sum()),
        "unfavourable_0": int((y == 0).sum()),
        "unfavourable_%": round((y == 0).mean() * 100, 2),
    })

data_summary = pd.DataFrame(summary_rows)
data_summary


## 3B. Exploratory Data Analysis figures

This section adds EDA figures before modelling. These figures help verify the processed week-cutoff datasets and give the report team visual evidence of class balance, VLE engagement over time, missingness, and simple target relationships.


In [ ]:
# EDA Figure 1: Mean VLE engagement by week
vle_summary_rows = []

for week in WEEKS:
    df = load_week_df(week)
    total_col = f"total_clicks_{week}"
    active_col = f"active_days_{week}"

    vle_summary_rows.append({
        "week": WEEK_NUM[week],
        "mean_total_clicks": float(df[total_col].mean()) if total_col in df.columns else np.nan,
        "mean_active_days": float(df[active_col].mean()) if active_col in df.columns else np.nan,
    })

vle_summary = pd.DataFrame(vle_summary_rows).sort_values("week")

plt.figure(figsize=(8, 5))
plt.plot(vle_summary["week"], vle_summary["mean_total_clicks"], marker="o")
plt.xlabel("Week cutoff")
plt.ylabel("Mean total clicks")
plt.title("Mean VLE total clicks across week cutoffs")
plt.xticks([2, 4, 6, 8])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/eda_mean_total_clicks_by_week.png", dpi=200)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(vle_summary["week"], vle_summary["mean_active_days"], marker="o")
plt.xlabel("Week cutoff")
plt.ylabel("Mean active days")
plt.title("Mean VLE active days across week cutoffs")
plt.xticks([2, 4, 6, 8])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/eda_mean_active_days_by_week.png", dpi=200)
plt.show()

vle_summary.to_csv("results/eda_vle_engagement_summary.csv", index=False)
print("Saved: results/eda_mean_total_clicks_by_week.png")
print("Saved: results/eda_mean_active_days_by_week.png")
print("Saved: results/eda_vle_engagement_summary.csv")
vle_summary


In [ ]:
# EDA table: Missing-value columns in Week 

Path("results").mkdir(exist_ok=True)

week_for_missing = "week4"
df_missing = load_week_df(week_for_missing)

missing_percent = (
    df_missing.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

top_missing = missing_percent[missing_percent > 0].head(15)

if len(top_missing) == 0:
    print(f"No missing values found in {week_for_missing}.")
    top_missing_df = pd.DataFrame(columns=["feature", "missing_percent"])
else:
    top_missing_df = top_missing.reset_index()
    top_missing_df.columns = ["feature", "missing_percent"]

top_missing_df.to_csv("results/eda_week4_missing_values.csv", index=False)

print("Saved: results/eda_week4_missing_values.csv")
top_missing_df


In [ ]:
# EDA Figure 3: Numeric feature correlation with target in Week 4
week_for_corr = "week4"
df_corr = load_week_df(week_for_corr)
X_corr, y_corr = split_xy(df_corr)

numeric_cols = X_corr.select_dtypes(include=["int64", "int32", "float64", "float32", "Int64"]).columns.tolist()

corr_rows = []
for col in numeric_cols:
    series = X_corr[col]
    if series.nunique(dropna=True) > 1:
        corr_value = series.corr(y_corr)
        if pd.notna(corr_value):
            corr_rows.append({
                "feature": col,
                "correlation_with_label": float(corr_value),
                "absolute_correlation": float(abs(corr_value)),
            })

corr_df = (
    pd.DataFrame(corr_rows)
    .sort_values("absolute_correlation", ascending=False)
    .head(15)
)

if len(corr_df) == 0:
    print("No numeric correlations available.")
else:
    plot_df = corr_df.sort_values("correlation_with_label")
    plt.figure(figsize=(9, 6))
    plt.barh(plot_df["feature"], plot_df["correlation_with_label"])
    plt.xlabel("Correlation with favourable outcome label")
    plt.ylabel("Numeric feature")
    plt.title("Top numeric feature correlations with target, Week 4")
    plt.grid(True, axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig("results/eda_week4_numeric_correlations.png", dpi=200)
    plt.show()
    print("Saved: results/eda_week4_numeric_correlations.png")

corr_df.to_csv("results/eda_week4_numeric_correlations.csv", index=False)
print("Saved: results/eda_week4_numeric_correlations.csv")
corr_df


In [ ]:
# EDA Figure 4: Favourable outcome rate by highest education, if available
week_for_category = "week4"
df_cat = load_week_df(week_for_category)

if "highest_education" not in df_cat.columns:
    print("highest_education column not found, skipping this EDA figure.")
else:
    edu_rate = (
        df_cat
        .groupby("highest_education")[TARGET]
        .mean()
        .sort_values()
        .mul(100)
        .reset_index()
    )
    edu_rate.columns = ["highest_education", "favourable_rate_percent"]

    plt.figure(figsize=(9, 5))
    plt.barh(edu_rate["highest_education"], edu_rate["favourable_rate_percent"])
    plt.xlabel("Favourable outcome rate (%)")
    plt.ylabel("Highest education")
    plt.title("Favourable outcome rate by highest education, Week 4")
    plt.grid(True, axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig("results/eda_week4_highest_education_outcome_rate.png", dpi=200)
    plt.show()

    edu_rate.to_csv("results/eda_week4_highest_education_outcome_rate.csv", index=False)
    print("Saved: results/eda_week4_highest_education_outcome_rate.png")
    print("Saved: results/eda_week4_highest_education_outcome_rate.csv")
    display(edu_rate)


## 4. View Task 4 baseline results and saved model inspection

In [ ]:
if task4_results_path is not None:
    task4_baseline_results = pd.read_csv(task4_results_path)
    display(task4_baseline_results.head())
else:
    task4_baseline_results = None
    print("No Task 4 results CSV found.")

if model_inspection_path is not None:
    model_inspection = pd.read_csv(model_inspection_path)
    display(model_inspection[[
        "file", "loaded_successfully", "object_type", "is_pipeline",
        "pipeline_steps", "has_predict", "has_predict_proba", "n_features_in", "load_error"
    ]])
else:
    model_inspection = None
    print("No model inspection summary found.")


## 5. Modelling helper functions

These helpers keep preprocessing inside the sklearn pipeline, so imputation, encoding, and scaling are fitted only on the training fold during CV.


In [ ]:
def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def make_bagging_logreg():
    base_model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    try:
        return BaggingClassifier(
            estimator=base_model,
            n_estimators=10,
            random_state=RANDOM_STATE,
        )
    except TypeError:
        return BaggingClassifier(
            base_estimator=base_model,
            n_estimators=10,
            random_state=RANDOM_STATE,
        )

def infer_columns(X: pd.DataFrame):
    categorical_cols = X.select_dtypes(include=["object", "bool", "category"]).columns.tolist()
    numeric_cols = X.select_dtypes(include=["int64", "int32", "float64", "float32", "Int64"]).columns.tolist()
    return categorical_cols, numeric_cols

def build_onehot_preprocessor(X: pd.DataFrame, scale_numeric=True):
    categorical_cols, numeric_cols = infer_columns(X)

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot_encoder()),
    ])

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_pipeline = Pipeline(numeric_steps)

    transformers = []
    if categorical_cols:
        transformers.append(("cat", categorical_pipeline, categorical_cols))
    if numeric_cols:
        transformers.append(("num", numeric_pipeline, numeric_cols))

    return ColumnTransformer(transformers=transformers, remainder="drop")

def build_ordinal_preprocessor(X: pd.DataFrame, scale_numeric=True):
    categorical_cols, numeric_cols = infer_columns(X)

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_pipeline = Pipeline(numeric_steps)

    transformers = []
    if categorical_cols:
        transformers.append(("cat", categorical_pipeline, categorical_cols))
    if numeric_cols:
        transformers.append(("num", numeric_pipeline, numeric_cols))

    return ColumnTransformer(transformers=transformers, remainder="drop")

def build_target_preprocessor(X: pd.DataFrame, scale_numeric=True):
    if not HAS_TARGET_ENCODER:
        raise RuntimeError("TargetEncoder is not available in this sklearn version.")

    categorical_cols, numeric_cols = infer_columns(X)

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("target", TargetEncoder(random_state=RANDOM_STATE)),
    ])

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_pipeline = Pipeline(numeric_steps)

    transformers = []
    if categorical_cols:
        transformers.append(("cat", categorical_pipeline, categorical_cols))
    if numeric_cols:
        transformers.append(("num", numeric_pipeline, numeric_cols))

    return ColumnTransformer(transformers=transformers, remainder="drop")

SCORING = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "auc": "roc_auc",
}

def select_best_threshold(y_true, probabilities):
    """Find the classification threshold that maximises F1 on validation data.
    
    Ported from OULAD_Team_4.ipynb so that task6 uses the same threshold logic
    as the main benchmark notebook.
    """
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
    if thresholds.size == 0:
        threshold = 0.5
        f1_val = f1_score(y_true, (probabilities >= threshold).astype(int),
                          zero_division=0)
        return float(threshold), float(f1_val)
    numerator   = 2 * precision[:-1] * recall[:-1]
    denominator = np.clip(precision[:-1] + recall[:-1], 1e-12, None)
    f1_scores   = numerator / denominator
    best_idx    = int(np.nanargmax(f1_scores))
    return float(thresholds[best_idx]), float(f1_scores[best_idx])


def evaluate_pipeline_cv(X, y, pipeline, cv):
    """5-fold stratified CV with per-fold F1-optimal threshold selection.

    For each fold:
      1. Hold out 20 % of the training fold to select the best-F1 threshold.
      2. Fit on the remaining 80 %.
      3. Evaluate on the test fold using that threshold.

    This matches the threshold logic in OULAD_Team_4.ipynb so that every
    reported metric — accuracy, precision, recall, F1, AUC — is computed at
    the same optimised threshold, not a fixed 0.5 cut-off.
    """
    fold_records = []
    for train_idx, test_idx in cv.split(X, y):
        X_tr_full = X.iloc[train_idx]
        y_tr_full = y.iloc[train_idx]
        X_te      = X.iloc[test_idx]
        y_te      = y.iloc[test_idx]

        # Hold out 20 % of training for threshold selection only
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_tr_full, y_tr_full,
            test_size=0.20,
            random_state=RANDOM_STATE,
            stratify=y_tr_full,
        )

        pipe = clone(pipeline)
        pipe.fit(X_tr, y_tr)

        # Select threshold on the held-out validation portion
        val_probs         = pipe.predict_proba(X_val)[:, 1]
        threshold, _      = select_best_threshold(y_val, val_probs)

        # Evaluate on the test fold
        test_probs  = pipe.predict_proba(X_te)[:, 1]
        test_preds  = (test_probs >= threshold).astype(int)

        fold_records.append({
            "accuracy":  float(accuracy_score(y_te, test_preds)),
            "precision": float(precision_score(y_te, test_preds, zero_division=0)),
            "recall":    float(recall_score(y_te, test_preds, zero_division=0)),
            "f1":        float(f1_score(y_te, test_preds, zero_division=0)),
            "auc":       float(roc_auc_score(y_te, test_probs)),
            "threshold": threshold,
        })

    df = pd.DataFrame(fold_records)
    record = {}
    for col in df.columns:
        record[f"{col}_mean"] = float(df[col].mean())
        record[f"{col}_std"]  = float(df[col].std(ddof=0))
    return record

def round_record(record, digits=4):
    return {k: round(v, digits) if isinstance(v, float) else v for k, v in record.items()}

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)


## 6. Official Task 4 baseline families, now evaluated with 5-fold CV

The latest Task 4 script uses six baseline model families:

- B1 Logistic Regression
- B2 L1-Regularised Logistic Regression
- B3 SGD Logistic
- B4 KNN
- B5 Gaussian Naive Bayes
- B6 Bagging Logistic Regression

The original Task 4 output used an 80/20 stratified split. In this notebook, the same model families are re-evaluated using 5-fold stratified CV for Task 6.


In [ ]:
def task4_baseline_model_factories():
    return {
        "B1_LogisticRegression": lambda: LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE,
        ),
        "B2_L1_Regularised_LogisticRegression": lambda: LogisticRegression(
            penalty="l1",
            solver="liblinear",
            max_iter=2000,
            random_state=RANDOM_STATE,
        ),
        "B3_SGD_Logistic": lambda: SGDClassifier(
            loss="log_loss",
            max_iter=2000,
            tol=1e-3,
            random_state=RANDOM_STATE,
        ),
        "B4_KNN": lambda: KNeighborsClassifier(
            n_neighbors=15,
        ),
        "B5_GaussianNB": lambda: GaussianNB(),
        "B6_Bagging_LogisticRegression": make_bagging_logreg,
    }

task4_cv_records = []

for week in WEEKS:
    df = load_week_df(week)
    X, y = split_xy(df)

    print(f"\n[{week.upper()}]")
    for model_name, factory in task4_baseline_model_factories().items():
        model = factory()
        preprocessor = build_onehot_preprocessor(X, scale_numeric=True)

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model),
        ])

        result = evaluate_pipeline_cv(X, y, pipeline, cv)
        print(
            f"{model_name:<42} "
            f"AUC={result['auc_mean']:.4f}±{result['auc_std']:.4f}  "
            f"F1={result['f1_mean']:.4f}  "
            f"thr={result['threshold_mean']:.3f}"
        )

        task4_cv_records.append({
            "week": week,
            "week_cutoff": WEEK_NUM[week],
            "pipeline_id": model_name,
            "model": model_name,
            "model_group": "Task 4 baseline",
            "feature_set": "Task 3 processed features",
            "preprocessing": "SimpleImputer + OneHotEncoder + StandardScaler",
            "evaluation_method": f"{CV_FOLDS}-fold stratified CV",
            "threshold": result["threshold_mean"],
            "seed": RANDOM_STATE,
            **round_record(result),
        })

task4_cv_results = pd.DataFrame(task4_cv_records)
task4_cv_results.to_csv("results/task6_task4_baseline_cv_results.csv", index=False)
print("\nSaved: results/task6_task4_baseline_cv_results.csv")
task4_cv_results


## 7. Task 6 baseline/advanced comparison models

These models are useful for pipeline comparison beyond the taught Task 4 baseline families.


In [ ]:
def extra_model_factories():
    factories = {
        "LR_weighted_onehot": lambda: (
            "SimpleImputer + OneHotEncoder + StandardScaler",
            build_onehot_preprocessor,
            LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
        ),
        "RandomForest_ordinal": lambda: (
            "SimpleImputer + OrdinalEncoder + StandardScaler",
            build_ordinal_preprocessor,
            RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
        "GradientBoosting_ordinal": lambda: (
            "SimpleImputer + OrdinalEncoder + StandardScaler",
            build_ordinal_preprocessor,
            GradientBoostingClassifier(
                n_estimators=200,
                random_state=RANDOM_STATE,
            ),
        ),
    }

    if HAS_CALIBRATION:
        factories["RF_calibrated_Task5_style"] = lambda: (
            "SimpleImputer + OrdinalEncoder + StandardScaler",
            build_ordinal_preprocessor,
            CalibratedClassifierCV(
                estimator=RandomForestClassifier(
                    n_estimators=100,
                    max_depth=10,
                    max_features="sqrt",
                    min_samples_leaf=2,
                    min_samples_split=5,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
                method="sigmoid",
                cv=5,
            ),
        )

    factories["GBM_Task5_style"] = lambda: (
        "SimpleImputer + OrdinalEncoder + StandardScaler",
        build_ordinal_preprocessor,
        GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=5,
            random_state=RANDOM_STATE,
        ),
    )

    factories["RF_weighted_Task5_style"] = lambda: (
        "SimpleImputer + OrdinalEncoder + StandardScaler",
        build_ordinal_preprocessor,
        RandomForestClassifier(
            n_estimators=100,
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    )

    factories["KNN_Task5_style"] = lambda: (
        "SimpleImputer + OneHotEncoder + StandardScaler",
        build_onehot_preprocessor,
        KNeighborsClassifier(n_neighbors=5),
    )

    if HAS_XGBOOST:
        factories["XGBoost_Task5_style"] = lambda: (
            "SimpleImputer + OrdinalEncoder + StandardScaler",
            build_ordinal_preprocessor,
            XGBClassifier(
                n_estimators=200,
                random_state=RANDOM_STATE,
                eval_metric="logloss",
                verbosity=0,
                n_jobs=-1,
            ),
        )

    if HAS_CATBOOST:
        factories["CatBoost_Task5_style"] = lambda: (
            "SimpleImputer + OrdinalEncoder + StandardScaler",
            build_ordinal_preprocessor,
            CatBoostClassifier(
                iterations=300,
                depth=6,
                learning_rate=0.05,
                loss_function="Logloss",
                eval_metric="AUC",
                auto_class_weights="Balanced",
                random_state=RANDOM_STATE,
                verbose=False,
            ),
        )

    return factories

extra_cv_records = []

for week in WEEKS:
    df = load_week_df(week)
    X, y = split_xy(df)

    print(f"\n[{week.upper()}]")
    for model_name, factory in extra_model_factories().items():
        preprocessing_name, preprocessor_factory, model = factory()
        preprocessor = preprocessor_factory(X, scale_numeric=True)

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model),
        ])

        result = evaluate_pipeline_cv(X, y, pipeline, cv)
        print(
            f"{model_name:<34} "
            f"AUC={result['auc_mean']:.4f}±{result['auc_std']:.4f}  "
            f"F1={result['f1_mean']:.4f}  "
            f"thr={result['threshold_mean']:.3f}"
        )

        extra_cv_records.append({
            "week": week,
            "week_cutoff": WEEK_NUM[week],
            "pipeline_id": model_name,
            "model": model_name,
            "model_group": "Task 6 / Task 5-style comparison",
            "feature_set": "Task 3 processed features",
            "preprocessing": preprocessing_name,
            "evaluation_method": f"{CV_FOLDS}-fold stratified CV",
            "threshold": result["threshold_mean"],
            "seed": RANDOM_STATE,
            **round_record(result),
        })

extra_cv_results = pd.DataFrame(extra_cv_records)
extra_cv_results.to_csv("results/task6_extra_model_cv_results.csv", index=False)
print("\nSaved: results/task6_extra_model_cv_results.csv")
extra_cv_results


## 8. Combined pipeline comparison table

In [ ]:
all_cv_results = pd.concat([task4_cv_results, extra_cv_results], ignore_index=True)
all_cv_results.to_csv("results/task6_all_cv_results.csv", index=False)

pipeline_comparison_table = all_cv_results[[
    "pipeline_id", "week_cutoff", "feature_set", "preprocessing",
    "model", "model_group", "evaluation_method", "threshold", "seed",
    "accuracy_mean", "precision_mean", "recall_mean", "f1_mean", "auc_mean",
    "accuracy_std", "precision_std", "recall_std", "f1_std", "auc_std",
]].copy()

pipeline_comparison_table.to_csv("results/task6_pipeline_comparison_table.csv", index=False)

best_pipeline_per_week = (
    all_cv_results
    .sort_values(["week_cutoff", "auc_mean", "f1_mean"], ascending=[True, False, False])
    .groupby("week_cutoff", as_index=False)
    .first()
)

best_pipeline_per_week["brief_takeaway"] = best_pipeline_per_week.apply(
    lambda row: f"{row['model']} achieved the highest mean AUC at week {row['week_cutoff']}.",
    axis=1,
)

best_pipeline_per_week.to_csv("results/task6_best_pipeline_per_week.csv", index=False)

print("Saved:")
print("- results/task6_all_cv_results.csv")
print("- results/task6_pipeline_comparison_table.csv")
print("- results/task6_best_pipeline_per_week.csv")

display(best_pipeline_per_week[[
    "week_cutoff", "pipeline_id", "model", "model_group",
    "accuracy_mean", "precision_mean", "recall_mean", "f1_mean", "auc_mean",
    "threshold", "brief_takeaway"
]])


## 8B. Table 2.4: Experiments table

This table is formatted to match the brief requirement more closely.

It explicitly includes:

- whether hyperparameters were changed,
- which metrics were computed,
- notes explaining the experiment purpose.


In [ ]:
def hyperparameter_changed_flag(pipeline_id: str) -> str:
    no_major_tuning = {
        "B1_LogisticRegression",
        "B5_GaussianNB",
    }
    return "No major tuning" if pipeline_id in no_major_tuning else "Yes"

def hyperparameter_notes(pipeline_id: str) -> str:
    notes = {
        "B1_LogisticRegression": "Mostly default Logistic Regression; max_iter and random_state set for convergence/reproducibility.",
        "B2_L1_Regularised_LogisticRegression": "L1 penalty and liblinear solver used.",
        "B3_SGD_Logistic": "SGD logistic model uses log_loss, max_iter, tolerance, and random_state.",
        "B4_KNN": "KNN uses n_neighbors=15.",
        "B5_GaussianNB": "GaussianNB used with default settings.",
        "B6_Bagging_LogisticRegression": "Bagging ensemble uses Logistic Regression base estimator and n_estimators=10.",
        "LR_weighted_onehot": "Logistic Regression uses class_weight='balanced'.",
        "RandomForest_ordinal": "Random Forest uses n_estimators=200 and class_weight='balanced'.",
        "GradientBoosting_ordinal": "Gradient Boosting uses n_estimators=200.",
        "RF_calibrated_Task5_style": "Random Forest wrapped in calibrated classifier using sigmoid calibration.",
        "GBM_Task5_style": "Gradient Boosting uses Task 5-style settings.",
        "RF_weighted_Task5_style": "Random Forest uses class_weight='balanced' and Task 5-style settings.",
        "KNN_Task5_style": "KNN uses Task 5-style n_neighbors setting.",
        "XGBoost_Task5_style": "XGBoost uses n_estimators and eval_metric settings.",
        "CatBoost_Task5_style": "CatBoost uses depth, learning rate, iterations, AUC eval metric, and balanced class weights.",
    }
    return notes.get(pipeline_id, "Model hyperparameters recorded in the notebook pipeline definition.")

def experiment_note(row) -> str:
    if row["model_group"] == "Task 4 baseline":
        return "Task 4 baseline family re-evaluated using 5-fold stratified CV."
    return "Task 5-style or Task 6 comparison model evaluated using the same CV protocol."

table_2_4_experiments = all_cv_results.copy()
table_2_4_experiments["Experiment ID"] = [
    f"EXP_{i+1:03d}" for i in range(len(table_2_4_experiments))
]
table_2_4_experiments["Hyperparameters changed?"] = table_2_4_experiments["pipeline_id"].apply(hyperparameter_changed_flag)
table_2_4_experiments["Hyperparameter notes"] = table_2_4_experiments["pipeline_id"].apply(hyperparameter_notes)
table_2_4_experiments["Metrics computed"] = "Accuracy, Precision, Recall, F1, AUC"
table_2_4_experiments["Notes"] = table_2_4_experiments.apply(experiment_note, axis=1)

table_2_4_experiments = table_2_4_experiments[[
    "Experiment ID",
    "week_cutoff",
    "pipeline_id",
    "model_group",
    "model",
    "feature_set",
    "evaluation_method",
    "threshold",
    "seed",
    "Hyperparameters changed?",
    "Hyperparameter notes",
    "Metrics computed",
    "Notes",
    "accuracy_mean",
    "precision_mean",
    "recall_mean",
    "f1_mean",
    "auc_mean",
]]

table_2_4_experiments.to_csv("results/task6_table_2_4_experiments.csv", index=False)
print("Saved: results/task6_table_2_4_experiments.csv")
table_2_4_experiments


## 8C. Table 2.5: Pipeline comparison table

This version separates **Encoding** and **Scaling** into their own columns instead of hiding them inside one preprocessing string.


In [ ]:
def extract_encoding(preprocessing: str) -> str:
    text = str(preprocessing)
    if "OneHotEncoder" in text:
        return "One-hot encoding"
    if "OrdinalEncoder" in text:
        return "Ordinal encoding"
    if "TargetEncoder" in text:
        return "Target encoding"
    return "Not specified"

def extract_scaling(preprocessing: str) -> str:
    text = str(preprocessing)
    if "StandardScaler" in text:
        return "StandardScaler"
    return "No scaling specified"

def extract_imputation(preprocessing: str) -> str:
    text = str(preprocessing)
    if "SimpleImputer" in text:
        return "SimpleImputer"
    return "Not specified"

table_2_5_pipeline_comparison = all_cv_results.copy()
table_2_5_pipeline_comparison["Encoding"] = table_2_5_pipeline_comparison["preprocessing"].apply(extract_encoding)
table_2_5_pipeline_comparison["Scaling"] = table_2_5_pipeline_comparison["preprocessing"].apply(extract_scaling)
table_2_5_pipeline_comparison["Imputation"] = table_2_5_pipeline_comparison["preprocessing"].apply(extract_imputation)
table_2_5_pipeline_comparison["Metrics computed"] = "Accuracy, Precision, Recall, F1, AUC"

table_2_5_pipeline_comparison = table_2_5_pipeline_comparison[[
    "pipeline_id",
    "week_cutoff",
    "feature_set",
    "model",
    "model_group",
    "Imputation",
    "Encoding",
    "Scaling",
    "evaluation_method",
    "threshold",
    "Metrics computed",
    "accuracy_mean",
    "precision_mean",
    "recall_mean",
    "f1_mean",
    "auc_mean",
    "accuracy_std",
    "precision_std",
    "recall_std",
    "f1_std",
    "auc_std",
]]

table_2_5_pipeline_comparison.to_csv("results/task6_table_2_5_pipeline_comparison.csv", index=False)
print("Saved: results/task6_table_2_5_pipeline_comparison.csv")
table_2_5_pipeline_comparison


## 9. Feature-drop ablation

Ablation asks: *what happens if we remove this feature group?*

A large negative AUC delta means the removed feature group is important.


In [ ]:
def vle_base_cols(week):
    return [f"total_clicks_{week}", f"active_days_{week}"]

def vle_activity_cols(week):
    return [f"{act}_clicks_{week}" for act in VLE_ACT_TYPES]

def existing(cols, X):
    return [c for c in cols if c in X.columns]

feature_ablation_records = []

for week in WEEKS:
    df = load_week_df(week)
    X_full, y = split_xy(df)

    demo_cols = existing(DEMO_CAT_COLS + DEMO_NUM_COLS, X_full)
    vle_base = existing(vle_base_cols(week), X_full)
    vle_activity = existing(vle_activity_cols(week), X_full)
    vle_all = vle_base + vle_activity

    configs = [
        ("baseline (all features)", X_full.columns.tolist()),
        ("drop student demographics", [c for c in X_full.columns if c not in demo_cols]),
        ("drop VLE features", [c for c in X_full.columns if c not in vle_all]),
        ("drop activity types — keep total + active_days", [c for c in X_full.columns if c not in vle_activity]),
        ("drop total + active_days — keep activity types", [c for c in X_full.columns if c not in vle_base]),
    ]

    for col in existing(DEMO_CAT_COLS + DEMO_NUM_COLS, X_full):
        configs.append((f"drop {col}", [c for c in X_full.columns if c != col]))

    print(f"\n[{week.upper()}]")
    for config_name, columns_used in configs:
        X = X_full[columns_used].copy()

        preprocessor = build_ordinal_preprocessor(X, scale_numeric=True)
        model = RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model),
        ])

        result = evaluate_pipeline_cv(X, y, pipeline, cv)
        print(f"{config_name:<55} AUC={result['auc_mean']:.4f}±{result['auc_std']:.4f}")

        feature_ablation_records.append({
            "week": week,
            "week_cutoff": WEEK_NUM[week],
            "experiment": "feature_drop",
            "configuration": config_name,
            "model": "Random Forest",
            "preprocessing": "SimpleImputer + OrdinalEncoder + StandardScaler",
            "evaluation_method": f"{CV_FOLDS}-fold stratified CV",
            "threshold": result["threshold_mean"],
            "seed": RANDOM_STATE,
            **round_record(result),
        })

feature_ablation = pd.DataFrame(feature_ablation_records)

# Add deltas relative to the baseline for each week.
baseline_lookup = (
    feature_ablation[feature_ablation["configuration"] == "baseline (all features)"]
    .set_index("week")
)

for metric in ["accuracy", "precision", "recall", "f1", "auc"]:
    feature_ablation[f"{metric}_delta"] = feature_ablation.apply(
        lambda row: round(row[f"{metric}_mean"] - baseline_lookup.loc[row["week"], f"{metric}_mean"], 4),
        axis=1,
    )

feature_ablation.to_csv("results/task6_feature_ablation_results.csv", index=False)
print("\nSaved: results/task6_feature_ablation_results.csv")
feature_ablation


## 10. Encoding ablation

In [ ]:
encoding_ablation_records = []
encoding_factories = {
    "ordinal": build_ordinal_preprocessor,
    "onehot": build_onehot_preprocessor,
}

if HAS_TARGET_ENCODER:
    encoding_factories["target"] = build_target_preprocessor

models_for_encoding = {
    "LogisticRegression_weighted": lambda: LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "RandomForest_weighted": lambda: RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

for week in WEEKS:
    df = load_week_df(week)
    X, y = split_xy(df)

    print(f"\n[{week.upper()}]")
    for encoding_name, preprocessor_factory in encoding_factories.items():
        for model_name, model_factory in models_for_encoding.items():
            preprocessor = preprocessor_factory(X, scale_numeric=True)
            model = model_factory()

            pipeline = Pipeline([
                ("preprocessor", preprocessor),
                ("model", model),
            ])

            result = evaluate_pipeline_cv(X, y, pipeline, cv)
            print(
                f"{model_name + ' / ' + encoding_name:<45} "
                f"AUC={result['auc_mean']:.4f}±{result['auc_std']:.4f}"
            )

            encoding_ablation_records.append({
                "week": week,
                "week_cutoff": WEEK_NUM[week],
                "experiment": "encoding",
                "configuration": f"{model_name} / {encoding_name}",
                "model": model_name,
                "encoding": encoding_name,
                "evaluation_method": f"{CV_FOLDS}-fold stratified CV",
                "threshold": result["threshold_mean"],
                "seed": RANDOM_STATE,
                **round_record(result),
            })

encoding_ablation = pd.DataFrame(encoding_ablation_records)
encoding_ablation.to_csv("results/task6_encoding_ablation_results.csv", index=False)
print("\nSaved: results/task6_encoding_ablation_results.csv")
encoding_ablation


## 11. Report ablation table

In [ ]:
report_ablation_table = feature_ablation[[
    "week_cutoff", "configuration", "model",
    "auc_mean", "auc_delta", "f1_mean", "f1_delta",
]].copy()

report_ablation_table["baseline_pipeline"] = "Random Forest baseline"
report_ablation_table["changed_pipeline"] = report_ablation_table["configuration"]
report_ablation_table["metric_impacted_most"] = "AUC"
report_ablation_table["result_direction_magnitude"] = report_ablation_table["auc_delta"].apply(lambda x: f"{x:+.4f}")

def interpret_ablation(row):
    config = row["configuration"]
    delta = row["auc_delta"]
    if config == "baseline (all features)":
        return "Reference pipeline for comparison."
    if "drop VLE features" in config:
        return "Large drop indicates VLE engagement is highly important."
    if "drop student demographics" in config:
        return "Demographics add useful information, but less than VLE behaviour."
    if abs(delta) < 0.005:
        return "Small change suggests limited individual impact."
    if delta < 0:
        return "Performance decreased, so the removed feature contributes useful signal."
    return "Performance did not decrease, suggesting the removed feature may be less useful."

report_ablation_table["interpretation"] = report_ablation_table.apply(interpret_ablation, axis=1)

report_ablation_table = report_ablation_table[[
    "week_cutoff", "baseline_pipeline", "changed_pipeline",
    "metric_impacted_most", "result_direction_magnitude",
    "auc_mean", "auc_delta", "f1_mean", "f1_delta", "interpretation",
]]

report_ablation_table.to_csv("results/task6_report_ablation_table.csv", index=False)
print("Saved: results/task6_report_ablation_table.csv")
report_ablation_table


## 12. Figures

In [ ]:
# Figure 1: AUC by week for all CV models
plot_df = all_cv_results.copy()

plt.figure(figsize=(11, 6))
for model_name, group in plot_df.groupby("pipeline_id"):
    group = group.sort_values("week_cutoff")
    plt.plot(group["week_cutoff"], group["auc_mean"], marker="o", label=model_name)

plt.xlabel("Week cutoff")
plt.ylabel("Mean AUC")
plt.title("Pipeline comparison: mean AUC by week cutoff")
plt.xticks([2, 4, 6, 8])
plt.grid(True, alpha=0.3)
plt.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig("results/task6_pipeline_auc_by_week.png", dpi=200)
plt.show()

print("Saved: results/task6_pipeline_auc_by_week.png")


In [ ]:
# Figure 2: Best model per week
best_plot = best_pipeline_per_week.sort_values("week_cutoff")

plt.figure(figsize=(8, 5))
plt.plot(best_plot["week_cutoff"], best_plot["auc_mean"], marker="o")
plt.xlabel("Week cutoff")
plt.ylabel("Best mean AUC")
plt.title("Best cross-validated AUC per week cutoff")
plt.xticks([2, 4, 6, 8])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/task6_best_auc_per_week.png", dpi=200)
plt.show()

print("Saved: results/task6_best_auc_per_week.png")


In [ ]:
# Figure 3: Week 4 feature-drop ablation
w4 = feature_ablation[
    (feature_ablation["week_cutoff"] == 4) &
    (feature_ablation["configuration"] != "baseline (all features)")
].sort_values("auc_delta")

plt.figure(figsize=(9, 6))
plt.barh(w4["configuration"], w4["auc_delta"])
plt.xlabel("AUC delta vs baseline")
plt.ylabel("Ablation configuration")
plt.title("Week 4 feature-drop ablation")
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("results/task6_week4_feature_ablation.png", dpi=200)
plt.show()

print("Saved: results/task6_week4_feature_ablation.png")


In [ ]:
# Figure 4: Encoding ablation
enc_plot = encoding_ablation.copy()

plt.figure(figsize=(10, 5))
for config, group in enc_plot.groupby("configuration"):
    group = group.sort_values("week_cutoff")
    plt.plot(group["week_cutoff"], group["auc_mean"], marker="o", label=config)

plt.xlabel("Week cutoff")
plt.ylabel("Mean AUC")
plt.title("Encoding ablation: mean AUC by week cutoff")
plt.xticks([2, 4, 6, 8])
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig("results/task6_encoding_ablation.png", dpi=200)
plt.show()

print("Saved: results/task6_encoding_ablation.png")


## 13. Written interpretation

In [ ]:
best_overall = all_cv_results.sort_values(["auc_mean", "f1_mean"], ascending=False).iloc[0]

largest_drop = (
    feature_ablation[feature_ablation["configuration"] != "baseline (all features)"]
    .sort_values("auc_delta")
    .iloc[0]
)

text = f'''
Cross-validation was performed using {CV_FOLDS}-fold stratified cross-validation across the week 2, 4, 6, and 8 feature tables. 
The strongest overall pipeline was {best_overall["pipeline_id"]} at week {best_overall["week_cutoff"]}, with a mean AUC of {best_overall["auc_mean"]:.4f} and a mean F1 score of {best_overall["f1_mean"]:.4f}. 
The results generally improved as the week cutoff increased, which suggests that additional student activity data improves early at-risk prediction.

The feature-drop ablation study showed that the largest AUC decrease occurred for "{largest_drop["configuration"]}" at week {largest_drop["week_cutoff"]}, with an AUC delta of {largest_drop["auc_delta"]:+.4f}. 
This indicates that the removed feature group carries important predictive information. 
In particular, decreases after removing VLE engagement features show that online activity behaviour is a major contributor to the prediction task.

The encoding ablation compared ordinal, one-hot, and target encoding where available. 
One-hot encoding is especially important for linear models because it avoids imposing artificial numerical order on categorical variables. 
Tree-based models were generally less sensitive to the encoding choice, although the final choice should still be based on the cross-validated AUC and F1 results.
'''

print(text)


# Return to Original Team 4 Notebook

The remaining cells continue from the original Team 4 notebook order.

## Reproducibility And Deliverables

The benchmark workflow is designed to produce outputs that can be reused in later stages of the project. With `SAVE_BENCHMARK_TABLES = True`, the comparison tables are written into `data/processed/`. If tuned model artifacts are also needed for inspection or deployment-style follow-up, the next cell can save them as well.

Set `SAVE_TRAINED_MODELS = True` in the config cell if you want to save the tuned artifacts. They are written into week-specific folders such as `models/retrained_week4/` and include the tuned estimator plus metadata such as threshold and feature columns.

In [ ]:
if SAVE_TRAINED_MODELS:
    for (week, model_name), artifact in trained_artifacts.items():
        output_dir = model_output_root / f'retrained_{week}'
        output_dir.mkdir(parents=True, exist_ok=True)

        payload = {
            'model': artifact['estimator'],
            'threshold': artifact['threshold'],
            'best_params': artifact['best_params'],
            'feature_columns': artifact['feature_columns'],
            'week': week,
        }
        if artifact['kind'] == 'catboost':
            payload['cat_features'] = artifact['cat_features']

        output_path = output_dir / model_name
        joblib.dump(payload, output_path)
        print(f'Saved {output_path}')
else:
    print('Saving is disabled. Set SAVE_TRAINED_MODELS = True in the config cell, rerun the benchmark cell, and then run this save cell.')

## Error Analysis

This section examines the prediction errors made by the best-performing model in each evaluated week. The analysis below focuses on the remaining mistakes made on the test set.

The purpose of this section is to identify whether the strongest models are more affected by false positives or false negatives, whether some student subgroups are more difficult to classify than others, and whether misclassified students show distinct engagement or background patterns. This helps move the evaluation beyond summary metrics and towards a more interpretable understanding of model behaviour.

In [ ]:
required_objects = ['best_models_per_week', 'trained_artifacts', 'week_splits']
missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise NameError(
        "Missing required benchmark objects: "
        + ", ".join(missing_objects)
        + ". Run the benchmark cell first."
    )

SUBGROUP_COLUMN = 'highest_education'   

error_frames_by_week = {}
subgroup_tables_by_week = {}
mistake_profiles_by_week = {}
summary_rows = []

for _, row in best_models_per_week.iterrows():
    week = row['week']
    model_name = row['model']

    artifact = trained_artifacts[(week, model_name)]
    split_data = week_splits[week]

    spec = {
        'kind': artifact['kind'],
        'feature_columns': artifact['feature_columns'],
        'cat_features': artifact.get('cat_features', []),
    }

    test_probabilities = predict_probabilities(
        spec,
        artifact['estimator'],
        split_data['X_test'],
    )
    test_predictions = (test_probabilities >= artifact['threshold']).astype(int)

    week_data = load_week_dataset(week).loc[split_data['X_test'].index].copy()
    week_data['y_true'] = split_data['y_test'].to_numpy()
    week_data['y_pred'] = test_predictions
    week_data['probability'] = test_probabilities
    week_data['threshold'] = artifact['threshold']

    week_data['error_type'] = np.select(
        [
            (week_data['y_true'] == 1) & (week_data['y_pred'] == 1),
            (week_data['y_true'] == 0) & (week_data['y_pred'] == 0),
            (week_data['y_true'] == 0) & (week_data['y_pred'] == 1),
            (week_data['y_true'] == 1) & (week_data['y_pred'] == 0),
        ],
        ['TP', 'TN', 'FP', 'FN'],
        default='Unknown',
    )

    week_data['distance_from_threshold'] = np.abs(
        week_data['probability'] - artifact['threshold']
    )

    error_frames_by_week[week] = week_data

    counts = week_data['error_type'].value_counts()
    tp = int(counts.get('TP', 0))
    tn = int(counts.get('TN', 0))
    fp = int(counts.get('FP', 0))
    fn = int(counts.get('FN', 0))

    summary_rows.append(
        {
            'week': week,
            'model': model_name,
            'threshold': round(float(artifact['threshold']), 3),
            'TP': tp,
            'TN': tn,
            'FP': fp,
            'FN': fn,
            'FP_rate': round(fp / len(week_data), 3),
            'FN_rate': round(fn / len(week_data), 3),
        }
    )

    if SUBGROUP_COLUMN in week_data.columns:
        subgroup_table = (
            week_data.groupby([SUBGROUP_COLUMN, 'error_type'])
            .size()
            .unstack(fill_value=0)
            .reindex(columns=['TP', 'TN', 'FP', 'FN'], fill_value=0)
        )

        subgroup_table['group_total'] = subgroup_table[['TP', 'TN', 'FP', 'FN']].sum(axis=1)
        subgroup_table['FP_rate'] = (subgroup_table['FP'] / subgroup_table['group_total']).round(3)
        subgroup_table['FN_rate'] = (subgroup_table['FN'] / subgroup_table['group_total']).round(3)

        subgroup_tables_by_week[week] = subgroup_table.sort_values(
            ['FN_rate', 'FP_rate'], ascending=False
        )

    key_features = [
        feature for feature in [
            f'total_clicks_{week}',
            f'active_days_{week}',
            'num_of_prev_attempts',
            'studied_credits',
        ]
        if feature in week_data.columns
    ]

    mistakes = week_data[week_data['error_type'].isin(['FP', 'FN'])].copy()

    if not mistakes.empty and key_features:
        mistake_profiles_by_week[week] = (
            mistakes.groupby('error_type')[key_features].median().round(2)
        )
    else:
        mistake_profiles_by_week[week] = pd.DataFrame()

summary_frame = pd.DataFrame(summary_rows).sort_values('week').reset_index(drop=True)

print('Best-model error counts by week')
display(summary_frame)

for week in summary_frame['week']:
    model_name = summary_frame.loc[summary_frame['week'] == week, 'model'].iloc[0]

    print(f'\n{week} - {model_name}')

    if week in subgroup_tables_by_week:
        print(f'Subgroup breakdown by {SUBGROUP_COLUMN}')
        display(subgroup_tables_by_week[week])

    if not mistake_profiles_by_week[week].empty:
        print('Median feature values for false positives and false negatives')
        display(mistake_profiles_by_week[week])


## Confusion Matrices, Accuracy / F1, and ROC / AUC Curves

This section provides detailed visual evaluation of the advanced models using confusion matrices and ROC / AUC curves. While the benchmark table already reports scalar metrics, confusion matrices make the distribution of true positives, true negatives, false positives, and false negatives immediately readable, and ROC curves show how the true-positive rate and false-positive rate trade against each other across the full range of classification thresholds. AUC then collapses that trade-off into a single ranking-quality score.

The section is split into two parts.

- **Part A** plots confusion matrices and ROC curves for the best-performing model at each of the four week cutoffs (week2, week4, week6, and week8). This highlights how predictive power improves as more engagement data accumulates.
- **Part B** compares all advanced models side-by-side at the latest cutoff (week8) using a shared ROC plot and a bar chart of Accuracy and F1. This makes it straightforward to see which model ranks best when the most student information is available.

All metrics are computed on the held-out test set using the threshold selected from the validation set in the benchmark section.

In [ ]:
# Part A — Confusion matrix and ROC / AUC for the best model at each week cutoff

required_objects = ['best_models_per_week', 'trained_artifacts', 'week_splits']
missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise NameError(
        'Missing required benchmark objects: '
        + ', '.join(missing_objects)
        + '. Run the benchmark cell before this section.'
    )

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
)

CM_LABELS = ['Favourable', 'Unfavourable']

for _, best_row in best_models_per_week.sort_values('week').iterrows():
    week = best_row['week']
    model_name = best_row['model']
    plot_name = model_name.replace('.pkl', '')

    artifact = trained_artifacts[(week, model_name)]
    split_data = week_splits[week]

    spec = {
        'kind': artifact['kind'],
        'feature_columns': artifact['feature_columns'],
        'cat_features': artifact.get('cat_features', []),
    }

    test_probabilities = predict_probabilities(spec, artifact['estimator'], split_data['X_test'])
    threshold = artifact['threshold']
    test_predictions = (test_probabilities >= threshold).astype(int)

    acc = accuracy_score(split_data['y_test'], test_predictions)
    f1 = f1_score(split_data['y_test'], test_predictions, zero_division=0)
    auc = roc_auc_score(split_data['y_test'], test_probabilities)
    fpr, tpr, _ = roc_curve(split_data['y_test'], test_probabilities)
    cm = confusion_matrix(split_data['y_test'], test_predictions)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CM_LABELS)
    disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
    axes[0].set_title(
        f'{week.upper()} — {plot_name}\n'
        f'Confusion Matrix  (threshold = {threshold:.3f})'
    )
    axes[0].set_xlabel('Predicted label')
    axes[0].set_ylabel('True label')

    axes[1].plot(fpr, tpr, color='tab:blue', lw=2, label=f'AUC = {auc:.3f}')
    axes[1].plot([0, 1], [0, 1], color='grey', linestyle='--', lw=1, label='Random classifier')
    axes[1].set_xlabel('False positive rate')
    axes[1].set_ylabel('True positive rate (recall)')
    axes[1].set_title(
        f'{week.upper()} — {plot_name}\n'
        f'ROC Curve  (Accuracy = {acc:.3f}, F1 = {f1:.3f})'
    )
    axes[1].grid(alpha=0.3)
    axes[1].legend(loc='lower right')

    plt.tight_layout()
    plt.show()

    print(f'{week.upper()} best model: {plot_name}')
    print(f'  Threshold : {threshold:.3f}')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  F1        : {f1:.4f}')
    print(f'  AUC       : {auc:.4f}')
    print()


In [ ]:
# Part B — ROC comparison and Accuracy / F1 bar chart for all advanced models at week8

COMPARISON_WEEK = 'week8'

dataset_w8 = load_week_dataset(COMPARISON_WEEK)
split_w8 = make_week_split(dataset_w8, COMPARISON_WEEK)
model_specs_w8 = build_model_specs(COMPARISON_WEEK, split_w8)

comparison_rows = []
roc_curves = {}

for model_name in BENCHMARK_MODELS:
    spec = model_specs_w8[model_name]
    best_estimator, best_params, _ = tune_estimator(spec, split_w8['X_train'], split_w8['y_train'])

    val_probs = predict_probabilities(spec, best_estimator, split_w8['X_val'])
    selected_threshold, _ = select_best_threshold(split_w8['y_val'], val_probs)

    test_probs = predict_probabilities(spec, best_estimator, split_w8['X_test'])
    test_preds = (test_probs >= selected_threshold).astype(int)

    acc = accuracy_score(split_w8['y_test'], test_preds)
    f1 = f1_score(split_w8['y_test'], test_preds, zero_division=0)
    auc = roc_auc_score(split_w8['y_test'], test_probs)
    fpr, tpr, _ = roc_curve(split_w8['y_test'], test_probs)

    short_name = model_name.replace('.pkl', '').replace('pipeline_', 'P').replace('_', ' ')
    comparison_rows.append({'model': short_name, 'Accuracy': acc, 'F1': f1, 'AUC': auc})
    roc_curves[short_name] = (fpr, tpr, auc)

comparison_frame = (
    pd.DataFrame(comparison_rows)
    .sort_values('AUC', ascending=False)
    .reset_index(drop=True)
)

print(f'Pipeline comparison at {COMPARISON_WEEK.upper()}')
display(comparison_frame.set_index('model').round(4))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for short_name, (fpr, tpr, auc) in roc_curves.items():
    axes[0].plot(fpr, tpr, lw=1.8, label=f'{short_name}  (AUC = {auc:.3f})')
axes[0].plot([0, 1], [0, 1], color='grey', linestyle='--', lw=1, label='Random classifier')
axes[0].set_xlabel('False positive rate')
axes[0].set_ylabel('True positive rate (recall)')
axes[0].set_title(f'{COMPARISON_WEEK.upper()} — ROC Curves: All Advanced Models')
axes[0].grid(alpha=0.3)
axes[0].legend(loc='lower right', fontsize=8)

x = np.arange(len(comparison_frame))
bar_width = 0.35
bars_acc = axes[1].bar(x - bar_width / 2, comparison_frame['Accuracy'], bar_width, label='Accuracy', color='tab:blue', alpha=0.85)
bars_f1 = axes[1].bar(x + bar_width / 2, comparison_frame['F1'], bar_width, label='F1', color='tab:orange', alpha=0.85)

for bar in bars_acc:
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f'{bar.get_height():.3f}',
        ha='center', va='bottom', fontsize=7,
    )
for bar in bars_f1:
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f'{bar.get_height():.3f}',
        ha='center', va='bottom', fontsize=7,
    )

axes[1].set_xticks(x)
axes[1].set_xticklabels(comparison_frame['model'], rotation=25, ha='right', fontsize=8)
axes[1].set_ylim(0, 1.08)
axes[1].set_ylabel('Score')
axes[1].set_title(f'{COMPARISON_WEEK.upper()} — Accuracy and F1: All Advanced Models')
axes[1].grid(axis='y', alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()
